# LA Studio voice-isolation - Spleeter 2-stem FP16

This notebook runs exactly `sherpa-onnx-spleeter-2stems-fp16` from the declared k2-fsa artifact on the temporary **Colab GPU worker**. It never uses API Gateway or a local LA Studio model.

The worker performs a CUDA startup probe before it prints a URL. It also sends long audio as bounded, overlapping segments, so the Spleeter FP16 CUDA convolution plan remains within the verified shape.

1. Choose **Runtime -> Change runtime type -> GPU**.
2. Run all cells. The final cell must print `startup probe: passed`.
3. Copy the printed URL and token to Dubbing -> Colab setup, then press **Check Colab**.


In [ ]:
!nvidia-smi
%pip install -q --upgrade --no-cache-dir "onnxruntime-gpu==1.21.0" "kaldi-native-fbank" "soundfile==0.13.1" "fastapi==0.115.12" "uvicorn==0.34.3" "python-multipart==0.0.20"

import torch
if not torch.cuda.is_available():
    raise RuntimeError('No Colab CUDA GPU is available. Select Runtime > Change runtime type > GPU, then restart and Run all.')
print('Colab CUDA:', torch.cuda.get_device_name(0))

!wget -q --show-progress -O /content/spleeter.tar.bz2 https://github.com/k2-fsa/sherpa-onnx/releases/download/source-separation-models/sherpa-onnx-spleeter-2stems-fp16.tar.bz2
!tar -xjf /content/spleeter.tar.bz2 -C /content


In [ ]:
from hashlib import sha256
from pathlib import Path
from urllib.request import urlopen

MODEL_ID = "sherpa-onnx-spleeter-2stems-fp16"
WORKER_COMMIT = "3c8f2bc3da273fc59e7a7aa5346dde41964de118"  # audited exact worker revision
WORKERS = {
    "la_studio_separation_worker.py": (
        "notebooks/workers/LA_STUDIO_SEPARATION_SPLEETER_2STEMS_WORKER.py",
        "307861926e13ff9849b04594074b573b1da063b1791c56dc2f502ab64991c5af"),
    "la_studio_separation_launcher.py": (
        "notebooks/workers/LA_STUDIO_SEPARATION_SPLEETER_2STEMS_LAUNCHER.py",
        "9ca893f8e06826bb875e68a7e364cdb43be515882b8438f01eb71465874375d1"),
}
for destination, (relative_path, expected_sha256) in WORKERS.items():
    url = f"https://raw.githubusercontent.com/khoinguyen59/kova-video-studio/{WORKER_COMMIT}/{relative_path}"
    payload = urlopen(url, timeout=60).read()
    actual_sha256 = sha256(payload).hexdigest()
    if actual_sha256 != expected_sha256:
        raise RuntimeError(f"Worker integrity check failed for {relative_path}: {actual_sha256}")
    Path('/content', destination).write_bytes(payload)
print('Downloaded verified exact-model CUDA worker templates.')


In [ ]:
!python /content/la_studio_separation_launcher.py


In [ ]:
# ==============================================================================
# 📥 LƯU FILE TRỰC TIẾP VÀO THƯ MỤC DỰ ÁN TRÊN MÁY TÍNH (FILE SYSTEM ACCESS API)
# ==============================================================================
import base64
import glob
import json
import os
from IPython.display import HTML, display

# Thu thập tất cả các file kết quả vừa tạo
result_files = {}
for pattern in ['/content/*.wav', '/content/*.srt', '/content/*.json', '/content/*/*/*.wav', '/content/*/*/*.srt']:
    for f in glob.glob(pattern):
        name = os.path.basename(f)
        if name not in result_files and os.path.isfile(f) and os.path.getsize(f) > 0:
            with open(f, 'rb') as fp:
                result_files[name] = base64.b64encode(fp.read()).decode('utf-8')

if not result_files:
    print("⚠️ Chưa có file kết quả mới để lưu.")
else:
    print(f"✅ Đã tìm thấy {len(result_files)} file kết quả: {', '.join(result_files.keys())}")
    print("👉 Bấm nút bên dưới và chọn thư mục 'LA-Studio/out/colab-live' để lưu thẳng vào máy:")
    
    files_json = json.dumps(result_files)
    html_code = f"""
    <button id="saveBtn" style="background-color: #2563eb; color: white; padding: 12px 24px; font-size: 15px; font-weight: bold; border: none; border-radius: 8px; cursor: pointer;">
        📁 Chọn Thư Mục & Lưu File Trực Tiếp Vào Máy
    </button>
    <div id="statusLog" style="margin-top: 10px; font-family: monospace; font-size: 13px; color: #16a34a;"></div>
    <script>
    document.getElementById('saveBtn').onclick = async () => {{
        const log = document.getElementById('statusLog');
        try {{
            if (!window.showDirectoryPicker) {{
                log.innerText = 'Trình duyệt không hỗ trợ File System Access API. Đang dùng tải thông thường...';
                return;
            }}
            log.innerText = 'Đang mở hộp thoại chọn thư mục...';
            const dirHandle = await window.showDirectoryPicker();
            const files = {files_json};
            for (const [name, b64] of Object.entries(files)) {{
                log.innerText = 'Đang ghi file: ' + name + '...';
                const fileHandle = await dirHandle.getFileHandle(name, {{ create: true }});
                const writable = await fileHandle.createWritable();
                const byteCharacters = atob(b64);
                const byteNumbers = new Array(byteCharacters.length);
                for (let i = 0; i < byteCharacters.length; i++) {{
                    byteNumbers[i] = byteCharacters.charCodeAt(i);
                }}
                const byteArray = new Uint8Array(byteNumbers);
                await writable.write(byteArray);
                await writable.close();
            }}
            log.innerText = '🎉 Đã lưu thành công toàn bộ file vào thư mục bạn chọn!';
        }} catch (err) {{
            if (err.name !== 'AbortError') {{
                log.innerText = 'Lỗi: ' + err.message;
            }} else {{
                log.innerText = 'Đã hủy chọn thư mục.';
            }}
        }}
    }};
    </script>
    """
    display(HTML(html_code))
